<h1>Setup. Organize XCP-D Outputs</h1>

Optional first step: parse an XCP-D output directory (`sub-*/ses-*/func`) and copy the pertinent files into a `RESULTS` directory:

```
RESULTS/atlases/{CIFTI,NIFTI}/<atlas>/{conn_mats,reho,timeseries}
RESULTS/motion/{dcan_qc,linc_qc}
```

`space-fsLR` files go to `CIFTI`; all other spaces go to `NIFTI`. Files are copied (never moved), re-runs skip identical files, and differing files are reported as `conflict` rather than overwritten. Set `RUN_SETUP = False` to skip this step when `RESULTS` already exists. Section 1 can then point at `RESULTS/atlases/NIFTI/<atlas>/conn_mats` and `RESULTS/motion/linc_qc` (or `dcan_qc`).

In [ ]:
from pathlib import Path
from IPython.display import display
from rsx_analyzer import DEFAULT_ATLASES, organize_xcpd_outputs

RUN_SETUP = False
XCPD_DIR = Path("/path/to/derivatives/xcp_d")
RESULTS_DIR = Path("/path/to/RESULTS")
ATLASES = list(DEFAULT_ATLASES)  # ["4S356", "Gordon", "HCP", "Tian"]

if RUN_SETUP:
    setup_result = organize_xcpd_outputs(XCPD_DIR, RESULTS_DIR, atlases=ATLASES)
    display(setup_result.summary)

<h1>Section 1. Loading Data</h1>

Configure the connectivity path, motion-QC path, and atlas, then run the package workflow. Motion processing mode is selected from the motion folder name (`dcan_qc` for HDF5 or `linc_qc` for CSV). Files are matched on subject, session, task, and run; the selected atlas is checked against matrix labels and order. Connectivity features include Schaefer 9/19-network summaries and left, right, and bilateral hippocampus/amygdala ROI-to-network measures.

### Motion metrics

| Metric | Meaning |
| --- | --- |
| DCAN: `mean_fd_remaining`, `frames_remaining`, `frames_total`, `frames_censored`, `seconds_remaining`, `fraction_remaining`, `skip` | HDF5 metrics at FD=0.3 |
| LINC: `mean_fd`, `num_censored_volumes`, `num_retained_volumes`, `space`, `res`, `desc` | Values read from the QC CSV |

Section 2 compares remaining scan duration at FD=0.3, 0.4, and 0.5 for DCAN QC, or summarizes LINC retained volumes. Run selection uses the highest remaining seconds (DCAN) or retained volumes (LINC). The 4S metadata assign cortical parcels to Schaefer networks and label subcortex/cerebellum explicitly.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "rsx_analyzer").is_dir()),
    Path.cwd(),
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from rsx_analyzer import (
    export_analysis_results_pdf,
    export_excel,
    plot_feature_motion_associations,
    plot_outlier_summary,
    plot_threshold_comparison,
    plot_triple_networks,
    robust_outlier_screen,
    run_workflow,
    summarize_multiple_runs,
    triple_network_summary,
)

CONN_MATS_PATH = Path("/Users/bustos/Documents/Projects/AKILI/REST/4S356/conn_mats")
MOTION_PATH = Path("/Users/bustos/Documents/Projects/AKILI/REST/linc_qc")
ATLAS_TSV_PATH = PROJECT_ROOT / "atlases" / "atlas-4S356Parcels_dseg.tsv"
TASK_FILTER = "rest"
FD_THRESHOLD = 0.3
OUTPUT_DIRECTORY = CONN_MATS_PATH.parent.parent
OUTPUT_EXCEL_PATH = OUTPUT_DIRECTORY / "rs-X1_analysis.xlsx"
OUTPUT_PDF_PATH = OUTPUT_DIRECTORY / "rs-X1_results.pdf"

In [ ]:
analysis = run_workflow(
    CONN_MATS_PATH,
    MOTION_PATH,
    ATLAS_TSV_PATH,
    task_filter=TASK_FILTER,
    fd_threshold=FD_THRESHOLD,
)

atlas = analysis.atlas
conn_files = analysis.discovery.conn_files
motion_files = analysis.discovery.motion_files
connectivity_df = analysis.connectivity.connectivity_df
feature_columns = list(analysis.connectivity.feature_columns)
motion_df = analysis.motion.motion_df
motion_load_errors_df = analysis.motion.motion_load_errors
run_level_df = analysis.run_level.run_level_df
coverage_summary = analysis.run_level.coverage_summary
missingness_report = analysis.run_level.missingness_report
motion_threshold_df = analysis.motion_threshold_data
threshold_summary = analysis.threshold_table
master_df = analysis.master.master_df
df_merged_runs = analysis.merged_runs
qc_motion_association = analysis.motion_associations

print(f"Atlas: {ATLAS_TSV_PATH.name} ({len(atlas.names)} parcels)")
print(f"Loaded {len(connectivity_df)} matrices and {len(motion_df)} readable motion files.")
print(f"Computed {len(feature_columns)} network and ROI-to-network features.")
print(f"Matched readable runs: {len(master_df)}; subject/session rows after run averaging: {len(df_merged_runs)}.")

In [ ]:
print("Subject coverage by session and run:")
display(coverage_summary)
print("Unmatched or unreadable subject/session/task/run files:")
display(missingness_report)
if not motion_load_errors_df.empty:
    print("Unreadable motion files:")
    display(motion_load_errors_df)

example_features = [
    "Cont_within_9net", "Cont_X_Default_9net", "ContA_within_19net",
    "ContA_X_DefaultA_19net", "LH_Hippocampus_to_Cont_9net",
    "RH_Hippocampus_to_Cont_9net", "Bilateral_Hippocampus_to_Cont_9net",
    "Bilateral_Amygdala_to_DefaultA_19net",
]
display(connectivity_df[["subject", "session", "task", "run", *example_features]].head())

<h1>Section 2. Quality Control</h1>

QC associations are exploratory and repeated runs/sessions are not independent observations.

### Run selection and session eligibility

Connectivity and motion records are first matched on subject, session, task, and run. `master_df` retains every matched readable run. To form `df_merged_runs`, runs within each subject/session are ranked by `remaining_seconds` (DCAN at the configured FD threshold) or `num_retained_volumes` (LINC), highest first; ties are ordered by run label. At most the top two runs are selected and averaged. The default minimum is 240: **every selected run** must have at least 240 seconds/retained volumes, or that subject/session is excluded from `df_merged_runs`. A session with only one matched run is retained when that run meets the minimum, including when its filename has no `run-` entity. The minimum can be changed with `minimum_retained_value` in `run_workflow`.

For LINC merged summaries, `num_censored_volumes` and `num_retained_volumes` are **sums across the selected runs**. `mean_fd`, other motion metrics, and connectivity features are averaged across those runs.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
if analysis.motion_mode == "dcan_qc":
    plot_threshold_comparison(motion_threshold_df, ax=ax)
    ax.set_xlabel("Framewise displacement threshold (mm)")
    ax.set_ylabel("Remaining scan duration (seconds)")
    ax.set_title("Remaining duration across motion thresholds")
else:
    ax.scatter(motion_threshold_df["mean_fd"], motion_threshold_df["num_retained_volumes"], alpha=0.6)
    ax.set_xlabel("Mean FD")
    ax.set_ylabel("Retained volumes")
    ax.set_title("LINC motion retention")
fig.tight_layout()
plt.show()
display(threshold_summary)

In [ ]:
print("Master run-level data:")
display(master_df.iloc[:5, :12])
print(f"Features most associated with {analysis.run_selection_column}:")
display(qc_motion_association[[
    "feature", "n_runs", "spearman_rho", "spearman_q", "pearson_r", "pearson_p"
]].head(20))
fig, ax = plt.subplots(figsize=(9, 7))
plot_feature_motion_associations(qc_motion_association, top_n=15, ax=ax)
ax.set_xlabel(f"Spearman correlation with {analysis.run_selection_column}")
ax.set_title("Connectivity features most associated with motion-related data loss")
fig.tight_layout()
plt.show()

multiple_runs_by_session, multiple_run_subjects = summarize_multiple_runs(df_merged_runs)
print("Subjects with multiple matched runs by session:")
display(multiple_runs_by_session)
print(f"Subjects with multiple runs in at least one session: {multiple_run_subjects['SubjectID'].nunique()}")
display(multiple_run_subjects)

<h2>Feature Outliers</h2>

Robust MAD/IQR scores flag extreme subject/session averages at |robust z| > 3.5. These are review flags, not automatic exclusions.

In [ ]:
OUTLIER_ROBUST_Z_THRESHOLD = 3.5
outlier_results = robust_outlier_screen(
    df_merged_runs,
    feature_columns,
    z_threshold=OUTLIER_ROBUST_Z_THRESHOLD,
)
outlier_feature_summary = outlier_results["feature_summary"]
outlier_feature_flags = outlier_results["feature_flags"]
outlier_subject_report = outlier_results["subject_report"]
unscorable_features = outlier_results["unscorable"]
print(
    f"Screened {len(feature_columns)} features across {len(df_merged_runs)} subject/session rows; "
    f"flagged {len(outlier_subject_report)} subjects in "
    f"{(outlier_feature_summary['n_outlier_rows'] > 0).sum()} features."
)
print(f"Unscorable features due to zero robust scale: {len(unscorable_features)}")
display(outlier_subject_report)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_outlier_summary(outlier_feature_summary, outlier_subject_report, axes=axes)
fig.tight_layout()
plt.show()

<h2>Triple-Network Connectivity</h2>

Compare within- and between-system connectivity for Control (CEN), Default (DMN), and Salience (SN). The 19-network values use parcel-pair-weighted summaries.

In [ ]:
TRIPLE_NETWORKS = {
    "Control (CEN)": {"network9": ("Cont",), "network19": ("ContA", "ContB", "ContC")},
    "Default (DMN)": {"network9": ("Default",), "network19": ("DefaultA", "DefaultB", "DefaultC")},
    "Salience (SN)": {"network9": ("SalVentAttn",), "network19": ("SalVentAttnA", "SalVentAttnB")},
}
network19_counts = pd.Series(
    atlas.networks["network_label_19network"]
).value_counts().to_dict()
triple_network_df, triple_summary = triple_network_summary(
    df_merged_runs,
    network19_counts,
    TRIPLE_NETWORKS,
)
triple_feature_order = [f"{name} within" for name in TRIPLE_NETWORKS] + [
    f"{left} x {right}"
    for index, left in enumerate(TRIPLE_NETWORKS)
    for right in list(TRIPLE_NETWORKS)[index + 1:]
]
display(triple_summary)
triple_ax = plot_triple_networks(triple_network_df, order=triple_feature_order)
triple_figure = triple_ax.figure
triple_ax.set_title("Triple-network connectivity by atlas resolution")
triple_figure.tight_layout()
plt.show()

<h2>Export Results</h2>

Write the analysis tables and print-ready QC report beside the input dataset.

In [ ]:
export_excel(
    {
        "master": master_df,
        "merged_runs": df_merged_runs,
        "QC": outlier_subject_report,
        "Trinetwork": triple_summary,
    },
    OUTPUT_EXCEL_PATH,
    figure=triple_figure,
    figure_sheet="Trinetwork",
    figure_title="Triple-network connectivity by atlas resolution",
    figure_subtitle="19-network summary uses parcel-pair-weighted means.",
)
export_analysis_results_pdf(
    OUTPUT_PDF_PATH,
    atlas_name=ATLAS_TSV_PATH.name,
    n_atlas_parcels=len(atlas.names),
    fd_threshold=FD_THRESHOLD,
    n_conn_files=len(conn_files),
    n_motion_files=len(motion_files),
    n_motion_readable=len(motion_df),
    n_master_rows=len(master_df),
    n_merged_run_rows=len(df_merged_runs),
    n_features=len(feature_columns),
    n_flagged_subjects=len(outlier_subject_report),
    threshold_summary=threshold_summary,
    motion_associations=qc_motion_association,
    outlier_feature_summary=outlier_feature_summary,
    outlier_subject_report=outlier_subject_report,
    triple_summary=triple_summary,
    triple_network_df=triple_network_df,
    triple_feature_order=triple_feature_order,
    outlier_z_threshold=OUTLIER_ROBUST_Z_THRESHOLD,
    motion_mode=analysis.motion_mode,
)
print(f"Excel workbook: {OUTPUT_EXCEL_PATH}")
print(f"Print-ready PDF: {OUTPUT_PDF_PATH}")